# Solar filament segmentation: detect, crop, segment, paint

This notebook runs the whole pipeline of our solution to the *Solar Filament Segmentation Challenge 2026*,
from the raw GONG H-alpha frames to the submission file:

1. **setup** and data location;
2. **pre-processing**: three enhanced input planes per frame;
3. **training** (optional, off by default): 5-fold box detector, self-training on unlabeled GONG frames, 5-fold crop segmenter;
4. **inference** on the 180 test frames with the released weights (`weights/` of this repository);
5. **post-processing**: confidence-ordered painting with the emission rule chosen on out-of-fold PQ, and the submission CSV;
6. **evaluation**: out-of-fold Panoptic Quality with the organisers' counting rule, IoU / Dice distributions and
   one-to-many / many-to-one relations;
7. **examples** of predicted masks.

It runs top to bottom on a Kaggle or Colab GPU (T4: about 30 minutes with `TRAIN = False`, `EVALUATE = False`) and on a
laptop without a GPU (slow). Nothing but the competition data and this repository is needed.

In [ ]:
TRAIN = False      # True: retrain every model from scratch (about 15 GPU-hours on a T4) instead of loading weights/
EVALUATE = False   # True: recompute the out-of-fold predictions and the emission rule (about 1 GPU-hour on a T4)

## 1. Setup

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = Path.cwd() if (Path.cwd() / "run.py").exists() else Path.cwd().parent if (Path.cwd().parent / "run.py").exists() else Path("solar-filament-segmentation-2026")
if not (REPO / "run.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/yauwes/solar-filament-segmentation-2026", str(REPO)], check=True)
REPO = REPO.resolve()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
# torch / torchvision come with Kaggle and Colab; requirements.txt pins the versions the weights were trained with
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.160", "segmentation-models-pytorch==0.5.0",
                "pycocotools", "scikit-image"], check=True)

def find_data():
    for base in (Path("/kaggle/input"), REPO / "data", Path("/content/data")):
        hits = sorted(base.rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json")) if base.exists() else []
        if hits:
            return hits[0].parents[1]
    raise FileNotFoundError("put MAGFiLO_1.0_Kaggle_2026/ (competition data) under data/")

os.environ["MAGFILO_ROOT"] = str(find_data())
os.environ["FILSEG_WORK"] = str(REPO / "work")
if not TRAIN:
    os.environ["FILSEG_WEIGHTS"] = str(REPO / "weights")          # released models
print("data:", os.environ["MAGFILO_ROOT"])

def run(*args):
    print("$ python run.py", *args, flush=True)
    subprocess.run([sys.executable, "run.py", *map(str, args)], check=True)

In [ ]:
import json
import cv2, numpy as np, matplotlib.pyplot as plt, torch
from pycocotools import mask as mu
from filseg import data, enhance, metric, postprocess, submission

recs, anns = data.coco_index()
print(f"train: {len(recs)} photos, {sum(map(len, recs.values()))} annotator records, {sum(map(len, anns.values()))} filaments | "
      f"test: {len(data.test_stems())} photos | device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Pre-processing

The detector sees the frame as it is. The crop segmenter sees three planes that each remove one nuisance and are
expressed in robust units (median / MAD inside the solar disk), so the six GONG sites land on a common scale:
a black top-hat (thin dark structures), the frame minus its 101-px median (limb darkening removed) and a Sato
ridge filter (elongated dark ridges).

In [ ]:
stem = data.split(0)[1][0]
img = data.read_gray(stem)
planes = enhance.enhance(img)
fig, ax = plt.subplots(1, 4, figsize=(16, 4.2))
for a, im, t in zip(ax, [img, *planes], ["H-alpha frame " + stem, *enhance.PLANES]):
    a.imshow(im[600:1400, 600:1400], cmap="gray"); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()

## 3. Training (optional)

Folds are grouped by observation date (`folds/folds.json`): frames of one day show the same filaments and never
sit on both sides of a split. Every model exists five times, each trained on four folds and validated on the fifth.

| step | command | T4 time per fold |
|---|---|---|
| detector (YOLO11m, 1280 px, boxes of every annotator record) | `run.py det --fold K --name m1280` | 1.7 h |
| self-training: the fold's detector labels 3,000 unlabeled GONG frames | `run.py pseudo --fold K --det m1280 --src DIR` | 0.8 h |
| student detector on real + pseudo labels | `run.py det --fold K --name m1280s --extra m1280 --epochs 20` | 2.8 h |
| crop segmenter (U-Net++ / EfficientNet-B4, 512-px crops) | `run.py seg --fold K --name e4` | 2.6 h |

The unlabeled frames are public GONG archive JPEGs (list and downloader: `scripts/gong_unlabeled.txt`,
`scripts/fetch_gong.py`). They carry no annotation; none is a competition frame.

In [ ]:
if TRAIN:
    run("planes")
    for k in range(5):
        run("det", "--fold", k, "--name", "m1280")
        run("seg", "--fold", k, "--name", "e4")
    if "m1280s" in "m1280":
        subprocess.run([sys.executable, "scripts/fetch_gong.py", "work/gong"], check=True)
        for k in range(5):
            run("pseudo", "--fold", k, "--det", "m1280", "--src", "work/gong")
            run("det", "--fold", k, "--name", "m1280s", "--extra", "m1280", "--epochs", 20)
print(sorted(p.name for p in data.WEIGHTS.glob("*.pt*")))

## 4. Inference on the test set

For every test frame: each detector sees the frame and its two flips, all boxes are fused by Weighted Boxes
Fusion; every box is cropped with 20% padding, the five segmenters and four flips are averaged; the result is
cached as one small file per frame (boxes, confidences, crop probabilities).

In [ ]:
run("predict", "--run", "main", "--det", "m1280", "--seg", "e4", "--test")

## 5. Post-processing and submission file

Instances are painted in confidence order; a pixel belongs to the first instance that claims it, so masks cannot
overlap. The emission rule (mask threshold, confidence floor, drop rule) was chosen once on out-of-fold PQ
(section 6) and is stored in `results/`.

In [ ]:
tune = REPO / "work" / "cache" / "main" / "tune.json"
if not tune.exists():                                   # the rule chosen on out-of-fold predictions, shipped with the repo
    tune.write_text((REPO / "results" / "tune_main.json").read_text())
run("submit", "--run", "main", "--out", "submission.csv")
print(json.load(open(tune))["cell"])

In [ ]:
# Same file as the one submitted to Kaggle? (GPU arithmetic is not bit-reproducible, so compare instance by instance.)
ref = REPO / "results" / "submission.csv"
if ref.exists():
    new, old = submission.read_csv("submission.csv"), submission.read_csv(ref)
    n = same = 0
    for s in sorted(set(new) | set(old)):
        a, b = new.get(s, []), old.get(s, [])
        n += max(len(a), len(b))
        if a and b:
            same += int((mu.iou(a, b, [0] * len(b)).max(1) > 0.9).sum())
    print(f"{sum(map(len, new.values()))} filaments now, {sum(map(len, old.values()))} in the submitted file; "
          f"{same} of {n} match at IoU > 0.9")

## 6. Evaluation

Panoptic Quality is counted exactly as in the organisers' self-evaluation notebook: a prediction and an annotation
match when IoU > 0.5; a frame annotated by several people is scored against each record and the counts are pooled.
All numbers are **out-of-fold**: every training frame is predicted by the models that did not see it.

In [ ]:
if EVALUATE:
    for k in range(5):
        run("predict", "--run", "main", "--det", "m1280", "--seg", "e4", "--fold", k)
    run("tune", "--run", "main")
res = json.load(open(tune))
print(f"out-of-fold PQ {res['pq']:.4f}  (95% CI {res['ci95'][0]:.4f} - {res['ci95'][1]:.4f}, dates resampled)   "
      f"SQ {res['sq']:.4f}  RQ {res['rq']:.4f}   TP {res['tp']}  FP {res['fp']}  FN {res['fn']}   on {res['photos']} photos")
print("per fold:", {k: round(v, 4) for k, v in res["per_fold"].items()})

In [ ]:
# Distributions the rubric asks for (IoU and Dice of overlapping pairs, one-to-many / many-to-one), out-of-fold.
stats = json.load(open(REPO / "results" / "overlap_stats.json"))
fig, ax = plt.subplots(1, 3, figsize=(16, 3.6))
for a, k in zip(ax, ("iou", "dice")):
    a.hist(stats[k], bins=50, range=(0, 1), color="navy", alpha=0.75)
    a.axvline(np.mean(stats[k]), color="crimson", ls="--", label=f"mean {np.mean(stats[k]):.3f}")
    a.set_xlabel(k.upper() if k == "iou" else "Dice"); a.legend()
deg = {k: np.bincount(np.minimum(stats[k], 3), minlength=4) for k in ("gt_degree", "pred_degree")}
x = np.arange(4)
ax[2].bar(x - 0.2, deg["gt_degree"] / deg["gt_degree"].sum(), 0.4, label="annotations: predictions touching it")
ax[2].bar(x + 0.2, deg["pred_degree"] / deg["pred_degree"].sum(), 0.4, label="predictions: annotations touching it")
ax[2].set_xticks(x, ["0 (missed / spurious)", "1 (one-to-one)", "2", "3+"]); ax[2].legend()
plt.tight_layout(); plt.show()

## 7. Examples

In [ ]:
pred = submission.read_csv("submission.csv")
stems = sorted(pred, key=lambda s: -len(pred[s]))[:3]
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
for a, s in zip(ax, stems):
    rgb = cv2.cvtColor(data.read_gray(s), cv2.COLOR_GRAY2RGB)
    for i, r in enumerate(pred[s]):
        cs, _ = cv2.findContours(mu.decode(r), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(rgb, cs, -1, [int(255 * v) for v in plt.get_cmap("tab10")(i % 10)[:3]], 3)
    a.imshow(rgb); a.set_title(f"{s}: {len(pred[s])} filaments"); a.axis("off")
plt.tight_layout(); plt.show()